# Predict Customer Churn
## Score: .91378

In [10]:
N_FOLDS = 5
N_SEEDS = 5
OPTUNA_TRIALS = 80
USE_RF_ET = False
USE_ORIGINAL_DATA = True
ORIGINAL_WEIGHT = 0.3
TARGET_ENC_M = 20
BLEND_MODE = 'rank'
USE_ISOTONIC_CALIBRATION = True
USE_CV_TARGET_ENCODING = True
STACK_RANK_LINEAR = False
USE_FEATURE_CLIP = False
USE_LOGREG_BLEND = False
USE_HGB = False
USE_DUAL_TRACK = False
USE_ORIG_FEATURES = True
USE_NGRAM_FEATURES = True
USE_LGB = False
USE_PSEUDO_LABELING = True
USE_STACKING = True
PSEUDO_WEIGHT = 0.2
PSEUDO_THRESHOLD_HIGH = 0.92
PSEUDO_THRESHOLD_LOW = 0.08
PSEUDO_MAX_PER_CLASS = 3000

In [11]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [ORIGINAL_WEIGHT] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [12]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    df['log_tenure'] = np.log1p(df['tenure'])
    df['TenureLow'] = (df['tenure'] < 12).astype(int)
    df['TenureMid'] = ((df['tenure'] >= 12) & (df['tenure'] < 36)).astype(int)
    df['TenureHigh'] = (df['tenure'] >= 36).astype(int)
    high_risk = (df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check') & (df['InternetService'] == 'Fiber optic')
    df['HighRiskCombo'] = high_risk.astype(int)
    df['FiberNoAddons'] = ((df['InternetService'] == 'Fiber optic') & (df['NumAddons'] == 0)).astype(int)
    df['MTM_ShortTenure'] = ((df['Contract'] == 'Month-to-month') & (df['tenure'] < 6)).astype(int)
    denom = 1 + df['NumAddons'] + df['NumStreaming']
    df['ChargePerService'] = df['MonthlyCharges'] / denom.replace(0, 1)
    df['ValueScore'] = (df['NumAddons'] + df['NumStreaming']) / (df['MonthlyCharges'] / 20 + 0.5)
    df['charges_deviation'] = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    svc_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['service_count'] = (df[svc_cols] == 'Yes').sum(axis=1).astype('float32')
    df['has_internet'] = (df['InternetService'] != 'No').astype('float32')
    df['has_phone'] = (df['PhoneService'] == 'Yes').astype('float32')
    df['tenure_years'] = (df['tenure'] // 12).astype('float32')
    df['tenure_mod12'] = (df['tenure'] % 12).astype('float32')
    df['mc_mod10'] = (np.floor(df['MonthlyCharges']) % 10).astype('float32')
    df['tc_mod100'] = (np.floor(df['TotalCharges']) % 100).astype('float32')
    t_str = df['tenure'].astype(str)
    df['tenure_first_digit'] = t_str.str[0].astype(int).astype('float32')
    df['tenure_last_digit'] = t_str.str[-1].astype(int).astype('float32')
    df['tc_rounded_100'] = (np.round(df['TotalCharges'] / 100) * 100).astype('float32')
    df['avg_monthly_from_total'] = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
    df['mc_fractional'] = (df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])).astype('float32')
    df['tc_fractional'] = (df['TotalCharges'] - np.floor(df['TotalCharges'])).astype('float32')
    df['log_TotalCharges'] = np.log1p(df['TotalCharges']).astype('float32')
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    freq = pd.concat([X_full[col], X_test_raw[col]]).value_counts(normalize=True)
    X_full[f'FREQ_{col}'] = X_full[col].map(freq).fillna(0).astype('float32')
    X_test_raw[f'FREQ_{col}'] = X_test_raw[col].map(freq).fillna(0).astype('float32')
for col, q in [('tenure', 6), ('MonthlyCharges', 5), ('TotalCharges', 5)]:
    _, bin_edges = pd.qcut(X_full[col], q=q, retbins=True, duplicates='drop')
    bin_edges = np.unique(bin_edges)
    bin_edges[0], bin_edges[-1] = -np.inf, np.inf
    X_full[f'{col}_bin'] = np.digitize(X_full[col].values, bin_edges[1:-1]).astype(str)
    X_test_raw[f'{col}_bin'] = np.digitize(X_test_raw[col].values, bin_edges[1:-1]).astype(str)

if USE_ORIG_FEATURES and USE_ORIGINAL_DATA:
    from itertools import combinations
    orig_prep = original.copy()
    orig_prep['Churn'] = orig_prep['Churn'].map({'Yes': 1, 'No': 0})
    orig_prep['TotalCharges'] = pd.to_numeric(orig_prep['TotalCharges'], errors='coerce')
    orig_prep['TotalCharges'] = orig_prep['TotalCharges'].fillna(orig_prep['TotalCharges'].median())
    for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling', 'OnlineSecurity', 'TechSupport', 'OnlineBackup', 'DeviceProtection']:
        tmp = orig_prep.groupby(col)['Churn'].mean()
        _name = f'ORIG_proba_{col}'
        X_full[_name] = X_full[col].map(tmp).fillna(0.5).astype('float32')
        X_test_raw[_name] = X_test_raw[col].map(tmp).fillna(0.5).astype('float32')
    for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
        tmp = orig_prep.groupby(col)['Churn'].mean()
        _name = f'ORIG_proba_{col}'
        X_full[_name] = X_full[col].map(tmp).fillna(0.5).astype('float32')
        X_test_raw[_name] = X_test_raw[col].map(tmp).fillna(0.5).astype('float32')
    def pctrank_against(values, ref):
        s = np.sort(ref)
        return (np.searchsorted(s, values) / len(s)).astype('float32')
    def zscore_against(values, ref):
        mu, sig = np.mean(ref), np.std(ref)
        return np.zeros(len(values), dtype='float32') if sig == 0 else ((values - mu) / sig).astype('float32')
    oc_tc = orig_prep.loc[orig_prep['Churn'] == 1, 'TotalCharges'].values
    onc_tc = orig_prep.loc[orig_prep['Churn'] == 0, 'TotalCharges'].values
    otc = orig_prep['TotalCharges'].values
    for df in [X_full, X_test_raw]:
        tc = df['TotalCharges'].values
        df['pctrank_nonchurner_TC'] = pctrank_against(tc, onc_tc)
        df['pctrank_churner_TC'] = pctrank_against(tc, oc_tc)
        df['pctrank_orig_TC'] = pctrank_against(tc, otc)
        df['zscore_churn_gap_TC'] = (np.abs(zscore_against(tc, oc_tc)) - np.abs(zscore_against(tc, onc_tc))).astype('float32')
        df['pctrank_churn_gap_TC'] = (pctrank_against(tc, oc_tc) - pctrank_against(tc, onc_tc)).astype('float32')
        df['zscore_nonchurner_TC'] = zscore_against(tc, onc_tc)
    orig_is_mc = orig_prep.groupby('InternetService')['MonthlyCharges'].mean()
    orig_pm_mc = orig_prep.groupby('PaymentMethod')['MonthlyCharges'].mean()
    for df in [X_full, X_test_raw]:
        df['resid_IS_MC'] = (df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc).fillna(0)).astype('float32')
        df['resid_PM_MC'] = (df['MonthlyCharges'] - df['PaymentMethod'].map(orig_pm_mc).fillna(0)).astype('float32')
        vals = np.zeros(len(df), dtype='float32')
        for cat_val in orig_prep['InternetService'].unique():
            mask = df['InternetService'] == cat_val
            ref = orig_prep.loc[orig_prep['InternetService'] == cat_val, 'TotalCharges'].values
            if len(ref) > 0 and mask.sum() > 0:
                vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
        df['cond_pctrank_IS_TC'] = vals
        vals = np.zeros(len(df), dtype='float32')
        for cat_val in orig_prep['Contract'].unique():
            mask = df['Contract'] == cat_val
            ref = orig_prep.loc[orig_prep['Contract'] == cat_val, 'TotalCharges'].values
            if len(ref) > 0 and mask.sum() > 0:
                vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
        df['cond_pctrank_C_TC'] = vals
    for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
        ch_q = np.quantile(oc_tc, q_val)
        nc_q = np.quantile(onc_tc, q_val)
        for df in [X_full, X_test_raw]:
            df[f'dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
            df[f'dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
            df[f'qdist_gap_To_{q_label}'] = (df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']).astype('float32')
    oc_mc = orig_prep.loc[orig_prep['Churn'] == 1, 'MonthlyCharges'].values
    onc_mc = orig_prep.loc[orig_prep['Churn'] == 0, 'MonthlyCharges'].values
    for df in [X_full, X_test_raw]:
        mc = df['MonthlyCharges'].values
        df['pctrank_churner_MC'] = pctrank_against(mc, oc_mc)
        df['pctrank_nonchurner_MC'] = pctrank_against(mc, onc_mc)
        df['zscore_churn_gap_MC'] = (np.abs(zscore_against(mc, oc_mc)) - np.abs(zscore_against(mc, onc_mc))).astype('float32')
    X_full['ORIG_proba_Contract_x_IS'] = (X_full['ORIG_proba_Contract'] * X_full['ORIG_proba_InternetService']).astype('float32')
    X_test_raw['ORIG_proba_Contract_x_IS'] = (X_test_raw['ORIG_proba_Contract'] * X_test_raw['ORIG_proba_InternetService']).astype('float32')
    X_full['ORIG_proba_Contract_x_PM'] = (X_full['ORIG_proba_Contract'] * X_full['ORIG_proba_PaymentMethod']).astype('float32')
    X_test_raw['ORIG_proba_Contract_x_PM'] = (X_test_raw['ORIG_proba_Contract'] * X_test_raw['ORIG_proba_PaymentMethod']).astype('float32')

if USE_NGRAM_FEATURES:
    from itertools import combinations
    TOP6 = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity', 'TechSupport', 'PaperlessBilling']
    TOP5 = TOP6[:5]
    NGRAM_COLS = []
    for c1, c2 in combinations(TOP6, 2):
        n = f'BG_{c1}_{c2}'
        NGRAM_COLS.append(n)
        X_full[n] = (X_full[c1].astype(str) + '_' + X_full[c2].astype(str))
        X_test_raw[n] = (X_test_raw[c1].astype(str) + '_' + X_test_raw[c2].astype(str))
    for c1, c2, c3 in combinations(TOP5, 3):
        n = f'TG_{c1}_{c2}_{c3}'
        NGRAM_COLS.append(n)
        X_full[n] = (X_full[c1].astype(str) + '_' + X_full[c2].astype(str) + '_' + X_full[c3].astype(str))
        X_test_raw[n] = (X_test_raw[c1].astype(str) + '_' + X_test_raw[c2].astype(str) + '_' + X_test_raw[c3].astype(str))

if USE_CV_TARGET_ENCODING:
    from sklearn.model_selection import StratifiedKFold
    cv_te = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
        X_full[f'{col}_te'] = np.nan
        for train_idx, val_idx in cv_te.split(X_full, y_full):
            tr = X_full.iloc[train_idx].join(y_full.iloc[train_idx])
            global_mean = tr['Churn'].mean()
            agg = tr.groupby(col)['Churn'].agg(['mean', 'count'])
            smooth = (agg['mean'] * agg['count'] + global_mean * TARGET_ENC_M) / (agg['count'] + TARGET_ENC_M)
            X_full.loc[val_idx, f'{col}_te'] = X_full.loc[val_idx, col].map(smooth).fillna(global_mean).values
        comb = pd.DataFrame({col: X_full[col], '_y': y_full.values})
        agg_full = comb.groupby(col)['_y'].agg(['mean', 'count'])
        smooth_full = (agg_full['mean'] * agg_full['count'] + y_full.mean() * TARGET_ENC_M) / (agg_full['count'] + TARGET_ENC_M)
        X_test_raw[f'{col}_te'] = X_test_raw[col].map(smooth_full).fillna(y_full.mean())
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])
else:
    for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
        X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])

if USE_NGRAM_FEATURES and NGRAM_COLS:
    from sklearn.model_selection import StratifiedKFold
    _cv_ng = cv_te if USE_CV_TARGET_ENCODING else StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for col in NGRAM_COLS:
        if col not in X_full.columns:
            continue
        X_full[f'TE_ng_{col}'] = np.nan
        for train_idx, val_idx in _cv_ng.split(X_full, y_full):
            tr = X_full.iloc[train_idx].join(y_full.iloc[train_idx])
            gm = tr['Churn'].mean()
            agg = tr.groupby(col, observed=False)['Churn'].mean()
            X_full.loc[val_idx, f'TE_ng_{col}'] = X_full.loc[val_idx, col].map(agg).fillna(gm).values
        comb_ng = pd.DataFrame({col: X_full[col], '_y': y_full.values})
        agg_ng = comb_ng.groupby(col, observed=False)['_y'].mean()
        X_test_raw[f'TE_ng_{col}'] = X_test_raw[col].map(agg_ng).fillna(y_full.mean())
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])

from sklearn.model_selection import StratifiedKFold
_cv_bin = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for col in ['tenure_bin', 'MonthlyCharges_bin', 'TotalCharges_bin']:
    if col not in X_full.columns:
        continue
    X_full[f'{col}_te'] = np.nan
    for train_idx, val_idx in _cv_bin.split(X_full, y_full):
        tr = X_full.iloc[train_idx].join(y_full.iloc[train_idx])
        gm = tr['Churn'].mean()
        agg = tr.groupby(col)['Churn'].mean()
        X_full.loc[val_idx, f'{col}_te'] = X_full.loc[val_idx, col].map(agg).fillna(gm).values
    comb_b = pd.DataFrame({col: X_full[col], '_y': y_full.values})
    agg_b = comb_b.groupby(col)['_y'].mean()
    X_test_raw[f'{col}_te'] = X_test_raw[col].map(agg_b).fillna(y_full.mean())
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

n_comp = len(train)
if USE_DUAL_TRACK and USE_ORIGINAL_DATA:
    X_comp_raw = engineer_features(train.drop(columns=['id', 'Churn']))
    tc_comp = X_comp_raw['TotalCharges'].median()
    X_test_comp_raw = engineer_features(test.drop(columns=['id']), total_charges_median=tc_comp)
    y_comp = train['Churn'].map({'Yes': 1, 'No': 0})
    if USE_CV_TARGET_ENCODING:
        for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
            X_comp_raw[f'{col}_te'] = np.nan
            for tr_idx, val_idx in cv_te.split(X_comp_raw, y_comp):
                tr = X_comp_raw.iloc[tr_idx].join(y_comp.iloc[tr_idx])
                gm = tr['Churn'].mean()
                agg = tr.groupby(col)['Churn'].agg(['mean', 'count'])
                smooth = (agg['mean'] * agg['count'] + gm * TARGET_ENC_M) / (agg['count'] + TARGET_ENC_M)
                X_comp_raw.loc[val_idx, f'{col}_te'] = X_comp_raw.loc[val_idx, col].map(smooth).fillna(gm).values
            comb = pd.DataFrame({col: X_comp_raw[col], '_y': y_comp.values})
            agg_full = comb.groupby(col)['_y'].agg(['mean', 'count'])
            smooth_full = (agg_full['mean'] * agg_full['count'] + y_comp.mean() * TARGET_ENC_M) / (agg_full['count'] + TARGET_ENC_M)
            X_test_comp_raw[f'{col}_te'] = X_test_comp_raw[col].map(smooth_full).fillna(y_comp.mean())
            X_comp_raw = X_comp_raw.drop(columns=[col])
            X_test_comp_raw = X_test_comp_raw.drop(columns=[col])
    else:
        for col in ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']:
            X_comp_raw, X_test_comp_raw = target_encode(X_comp_raw, X_test_comp_raw, col, y_comp, m=TARGET_ENC_M)
            X_comp_raw = X_comp_raw.drop(columns=[col])
            X_test_comp_raw = X_test_comp_raw.drop(columns=[col])
    X_comp_encoded = pd.get_dummies(X_comp_raw, drop_first=True)
    X_test_comp_encoded = pd.get_dummies(X_test_comp_raw, drop_first=True)
    X_test_comp_encoded = X_test_comp_encoded.reindex(columns=X_comp_encoded.columns, fill_value=0)
else:
    X_comp_encoded = None
    X_test_comp_encoded = None
    y_comp = None

if USE_FEATURE_CLIP:
    num_cols = X_encoded.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        lo, hi = X_encoded[c].quantile(0.01), X_encoded[c].quantile(0.99)
        X_encoded[c] = X_encoded[c].clip(lo, hi)
        X_test_encoded[c] = X_test_encoded[c].clip(lo, hi)

C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_31544\1506703802.py:160: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_full[n] = (X_full[c1].astype(str) + '_' + X_full[c2].astype(str))
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_31544\1506703802.py:161: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_raw[n] = (X_test_raw[c1].astype(str) + '_' + X_test_raw[c2].astype(str))
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_31544\1506703802.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is us

In [13]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [14]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-18 22:40:29,230] A new study created in memory with name: no-name-e7cf8b52-ed2b-4c71-970a-650047d05f9c


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-18 22:42:38,773] Trial 0 finished with value: 0.9140089596235551 and parameters: {'n_estimators': 474, 'max_depth': 7, 'learning_rate': 0.14830825324755828, 'subsample': 0.8206502089395048, 'colsample_bytree': 0.8792494118127986, 'scale_pos_weight': 2.6452433407694027, 'reg_alpha': 0.8713227527854734, 'reg_lambda': 0.21268401096029713, 'min_child_weight': 7}. Best is trial 0 with value: 0.9140089596235551.
[I 2026-03-18 22:44:30,991] Trial 1 finished with value: 0.9158675406625842 and parameters: {'n_estimators': 313, 'max_depth': 8, 'learning_rate': 0.02609329511762675, 'subsample': 0.710609339486399, 'colsample_bytree': 0.922844250679036, 'scale_pos_weight': 2.4179646280249685, 'reg_alpha': 0.04212646346196249, 'reg_lambda': 8.093761431926314, 'min_child_weight': 1}. Best is trial 1 with value: 0.9158675406625842.
[I 2026-03-18 22:46:46,999] Trial 2 finished with value: 0.9164117275440209 and parameters: {'n_estimators': 507, 'max_depth': 6, 'learning_rate': 0.035026058741

In [15]:
if USE_LGB:
    from lightgbm import LGBMClassifier

    def lgb_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        }
        scores = []
        for train_idx, val_idx in cv.split(X_encoded, y_full):
            X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
            y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
            sw_tr = sample_weights[train_idx]
            m = LGBMClassifier(**params, random_state=42, verbose=-1)
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        return np.mean(scores)

    study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
    study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_lgb_params = study_lgb.best_params
    print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")
else:
    best_lgb_params = {}

In [16]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-19 01:37:58,413] A new study created in memory with name: no-name-c0b6287c-8847-4590-acac-328feb80ba99


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-19 01:40:11,549] Trial 0 finished with value: 0.9153561816182364 and parameters: {'iterations': 604, 'depth': 3, 'learning_rate': 0.05641384196414387, 'subsample': 0.7286702618955773, 'colsample_bylevel': 0.7501673505243787, 'scale_pos_weight': 2.6322724467194503, 'l2_leaf_reg': 0.002451011392845066}. Best is trial 0 with value: 0.9153561816182364.
[I 2026-03-19 01:41:43,952] Trial 1 finished with value: 0.9164367911950453 and parameters: {'iterations': 309, 'depth': 5, 'learning_rate': 0.13495733161070828, 'subsample': 0.6462360032311989, 'colsample_bylevel': 0.7565898361177056, 'scale_pos_weight': 3.9892060605103623, 'l2_leaf_reg': 0.4175482699146762}. Best is trial 1 with value: 0.9164367911950453.
[I 2026-03-19 01:47:39,302] Trial 2 finished with value: 0.916923570512521 and parameters: {'iterations': 776, 'depth': 7, 'learning_rate': 0.06415047583262602, 'subsample': 0.8816502668783057, 'colsample_bylevel': 0.8019976105867572, 'scale_pos_weight': 1.8924338267640541, 'l2

In [17]:
if USE_HGB:
    from sklearn.ensemble import HistGradientBoostingClassifier

    def hgb_objective(trial):
        params = {
            'max_iter': trial.suggest_int('max_iter', 200, 600),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
            'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        }
        scores = []
        for train_idx, val_idx in cv.split(X_encoded, y_full):
            X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
            y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
            sw_tr = sample_weights[train_idx]
            m = HistGradientBoostingClassifier(**params, random_state=42)
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        return np.mean(scores)

    study_hgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
    study_hgb.optimize(hgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_hgb_params = study_hgb.best_params
    print(f"HGB Best CV AUC: {study_hgb.best_value:.4f}")
else:
    best_hgb_params = {}

In [18]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize
from scipy.stats import rankdata

n_models = (4 if USE_HGB else (2 if not USE_LGB else 3)) if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))
oof_comp = np.zeros(n_comp) if (USE_DUAL_TRACK and X_comp_encoded is not None) else None
test_comp = np.zeros(len(X_test_encoded)) if (USE_DUAL_TRACK and X_comp_encoded is not None) else None

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1) if USE_LGB else None,
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    models = [m for m in models if m is not None]
    if USE_HGB:
        models.append(HistGradientBoostingClassifier(**best_hgb_params, random_state=seed+fold))
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

n_orig = len(X_encoded)
fold_splits = list(cv.split(X_encoded, y_full))

def run_training_pass(X_use, y_use, sw_use, pseudo_train_idx=None):
    """Run one training pass. pseudo_train_idx=None for pass 0; for pass 1 pass indices of pseudo rows to add to train."""
    oof_pass = np.zeros((n_orig, n_models))
    test_preds_pass = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(fold_splits):
        if pseudo_train_idx is not None:
            train_idx_aug = np.concatenate([train_idx, pseudo_train_idx])
        else:
            train_idx_aug = train_idx
        X_tr = X_use.iloc[train_idx_aug] if hasattr(X_use, 'iloc') else X_use.loc[train_idx_aug]
        X_val = X_use.iloc[val_idx] if hasattr(X_use, 'iloc') else X_use.loc[val_idx]
        y_tr = y_use.iloc[train_idx_aug] if hasattr(y_use, 'iloc') else y_use[train_idx_aug]
        sw_tr = sw_use[train_idx_aug]
        models = get_models(42, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            oof_pass[val_idx, i] = m.predict_proba(X_val)[:, 1]
            test_preds_pass[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    return oof_pass, test_preds_pass

oof, test_preds = run_training_pass(X_encoded, y_full, sample_weights, pseudo_train_idx=None)

X_tr_seed, y_tr_seed, sw_tr_seed = X_encoded, y_full, sample_weights
pseudo_train_idx_seed = None

if USE_PSEUDO_LABELING:
    def _temp_blend(oof_arr, test_arr):
        r = np.column_stack([(rankdata(oof_arr[:, i]) - 0.5) / len(oof_arr) for i in range(n_models)])
        best_auc, best_w = -1, np.ones(n_models) / n_models
        for x0 in [best_w, np.array([0.5, 0.5] if n_models==2 else [0.33]*3)]:
            if len(x0) != n_models: continue
            from scipy.optimize import minimize
            res = minimize(lambda w: -roc_auc_score(y_full, r @ w), x0=x0, method='SLSQP',
                          bounds=[(0,1)]*n_models, constraints={'type':'eq','fun':lambda w: np.sum(w)-1})
            if -res.fun > best_auc: best_auc, best_w = -res.fun, res.x
        tr = np.column_stack([(rankdata(test_arr[:, i]) - 0.5) / len(test_arr) for i in range(n_models)])
        return np.clip(tr @ best_w, 0, 1)
    test_blend = _temp_blend(oof, test_preds)
    idx_high = np.where(test_blend >= PSEUDO_THRESHOLD_HIGH)[0][:PSEUDO_MAX_PER_CLASS]
    idx_low = np.where(test_blend <= PSEUDO_THRESHOLD_LOW)[0][:PSEUDO_MAX_PER_CLASS]
    pseudo_idx = np.concatenate([idx_high, idx_low])
    pseudo_labels = np.concatenate([np.ones(len(idx_high)), np.zeros(len(idx_low))])
    X_aug = pd.concat([X_encoded, X_test_encoded.iloc[pseudo_idx]], ignore_index=True)
    y_aug = pd.Series(np.concatenate([y_full.values, pseudo_labels]))
    sw_aug = np.concatenate([sample_weights, np.full(len(pseudo_idx), PSEUDO_WEIGHT)])
    pseudo_train_indices = np.arange(n_orig, n_orig + len(pseudo_idx))
    oof, test_preds = run_training_pass(X_aug, y_aug, sw_aug, pseudo_train_idx=pseudo_train_indices)
    print(f"Pseudo-labeled {len(pseudo_idx)} test samples ({len(idx_high)} churn, {len(idx_low)} no-churn)")
    X_tr_seed, y_tr_seed, sw_tr_seed = X_aug, y_aug, sw_aug
    pseudo_train_idx_seed = pseudo_train_indices

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    if oof_comp is not None:
        comp_tr = train_idx[train_idx < n_comp]
        comp_val = val_idx[val_idx < n_comp]
        if len(comp_tr) > 0 and len(comp_val) > 0:
            m_comp = LGBMClassifier(**best_lgb_params, random_state=42+fold, verbose=-1)
            m_comp.fit(X_comp_encoded.iloc[comp_tr], y_comp.iloc[comp_tr])
            oof_comp[comp_val] = m_comp.predict_proba(X_comp_encoded.iloc[comp_val])[:, 1]
            test_comp += m_comp.predict_proba(X_test_comp_encoded)[:, 1] / N_FOLDS

def to_rank_percentiles(arr):
    return (rankdata(arr) - 0.5) / len(arr)

oof_rank = np.column_stack([to_rank_percentiles(oof[:, i]) for i in range(n_models)])
oof_linear = oof.copy()

def neg_auc(w):
    blend = oof_blend_input @ w
    return -roc_auc_score(y_full, blend)

if STACK_RANK_LINEAR:
    oof_blend_input = oof_rank
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.5] if n_models==2 else [0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_rank = best_w
    oof_blend_input = oof_linear
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.5] if n_models==2 else [0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_linear = best_w
    blend_rank_oof = oof_rank @ w_rank
    blend_linear_oof = oof_linear @ w_linear
    best_auc, best_alpha = -1, 0.5
    for a in np.linspace(0, 1, 21):
        blended = a * blend_rank_oof + (1 - a) * blend_linear_oof
        auc = roc_auc_score(y_full, blended)
        if auc > best_auc:
            best_auc, best_alpha = auc, a
    blend_oof = best_alpha * blend_rank_oof + (1 - best_alpha) * blend_linear_oof
    blend_weights = best_alpha * w_rank + (1 - best_alpha) * w_linear
    stack_alpha = best_alpha
else:
    if BLEND_MODE == 'rank':
        oof_blend_input = oof_rank
    else:
        oof_blend_input = oof.copy()
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.5] if n_models==2 else [0.5, 0.3, 0.2] if n_models==3 else [0.25]*4 if n_models==4 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    blend_weights = best_w
    blend_oof = oof_blend_input @ blend_weights
    stack_alpha = None

dual_track_w = 1.0
if oof_comp is not None and USE_DUAL_TRACK:
    comp_idx = np.arange(n_comp)
    best_auc, best_w = -1, 1.0
    for w in np.linspace(0, 1, 21):
        blend_dual = w * blend_oof[comp_idx] + (1 - w) * oof_comp
        auc = roc_auc_score(y_comp, blend_dual)
        if auc > best_auc:
            best_auc, best_w = auc, w
    dual_track_w = best_w
    blend_oof = np.concatenate([dual_track_w * blend_oof[comp_idx] + (1 - dual_track_w) * oof_comp, blend_oof[n_comp:]])

use_logreg_blend = False
logreg_blender = None
if USE_LOGREG_BLEND and not STACK_RANK_LINEAR:
    from sklearn.linear_model import LogisticRegression
    logreg_blender = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    logreg_blender.fit(oof_rank, y_full)
    logreg_oof = logreg_blender.predict_proba(oof_rank)[:, 1]
    auc_scipy = roc_auc_score(y_full, blend_oof)
    auc_logreg = roc_auc_score(y_full, logreg_oof)
    if auc_logreg > auc_scipy:
        use_logreg_blend = True
        blend_oof = logreg_oof

use_ridge_blend = False
ridge_blender = None
if USE_STACKING and not use_logreg_blend and not STACK_RANK_LINEAR:
    from sklearn.linear_model import Ridge
    ridge_blender = Ridge(alpha=1.0, random_state=42)
    ridge_blender.fit(oof_rank, y_full)
    ridge_oof = np.clip(ridge_blender.predict(oof_rank), 0, 1)
    auc_ridge = roc_auc_score(y_full, ridge_oof)
    auc_current = roc_auc_score(y_full, blend_oof)
    if auc_ridge > auc_current:
        use_ridge_blend = True
        blend_oof = ridge_oof

if USE_ISOTONIC_CALIBRATION:
    iso_calib = IsotonicRegression(out_of_bounds='clip')
    iso_calib.fit(blend_oof, y_full)
    blend_oof = np.clip(iso_calib.predict(blend_oof), 0, 1)

names = ['XGB'] + (['LGB'] if USE_LGB else []) + ['Cat'] + (['HGB'] if USE_HGB else []) + (['RF','ET'] if USE_RF_ET else [])
mode_str = 'rank+linear' if STACK_RANK_LINEAR else ('logreg' if use_logreg_blend else ('ridge' if use_ridge_blend else BLEND_MODE))
print(f"Blend OOF AUC ({mode_str}): {roc_auc_score(y_full, blend_oof):.4f}")
if not use_logreg_blend and not use_ridge_blend:
    print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")
if oof_comp is not None and USE_DUAL_TRACK:
    print(f"Dual track (main weight): {dual_track_w:.3f}")
if STACK_RANK_LINEAR:
    print(f"Stack alpha (rank): {stack_alpha:.3f}")

Pseudo-labeled 6000 test samples (3000 churn, 3000 no-churn)
Blend OOF AUC (ridge): 0.9110


In [19]:
def blend_test(tp, apply_cal=True):
    if use_logreg_blend:
        tp_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = logreg_blender.predict_proba(tp_rank)[:, 1]
    elif use_ridge_blend:
        tp_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = np.clip(ridge_blender.predict(tp_rank), 0, 1)
    elif STACK_RANK_LINEAR:
        blend_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)]) @ w_rank
        blend_linear = tp @ w_linear
        preds = stack_alpha * blend_rank + (1 - stack_alpha) * blend_linear
    elif BLEND_MODE == 'rank':
        tp_input = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = tp_input @ blend_weights
    else:
        preds = tp @ blend_weights
    if apply_cal and USE_ISOTONIC_CALIBRATION:
        preds = np.clip(iso_calib.predict(preds), 0, 1)
    return preds

store_raw = (oof_comp is not None and USE_DUAL_TRACK)
all_test_preds = [blend_test(test_preds, apply_cal=not store_raw)]
all_test_comp = [test_comp.copy()] if test_comp is not None else None
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    tc = np.zeros(len(X_test_encoded)) if test_comp is not None else None
    for fold, (train_idx, val_idx) in enumerate(fold_splits):
        if pseudo_train_idx_seed is not None:
            train_idx_aug = np.concatenate([train_idx, pseudo_train_idx_seed])
        else:
            train_idx_aug = train_idx
        X_tr = X_tr_seed.iloc[train_idx_aug]
        y_tr = y_tr_seed.iloc[train_idx_aug]
        sw_tr = sw_tr_seed[train_idx_aug]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
        if tc is not None:
            comp_tr = train_idx[train_idx < n_comp]
            comp_val = val_idx[val_idx < n_comp]
            if len(comp_tr) > 0 and len(comp_val) > 0:
                m_comp = LGBMClassifier(**best_lgb_params, random_state=base_seed+fold, verbose=-1)
                m_comp.fit(X_comp_encoded.iloc[comp_tr], y_comp.iloc[comp_tr])
                tc += m_comp.predict_proba(X_test_comp_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(blend_test(tp, apply_cal=not store_raw))
    if all_test_comp is not None and tc is not None:
        all_test_comp.append(tc)

if all_test_comp is not None and store_raw:
    final_main_raw = np.mean(all_test_preds, axis=0)
    final_comp = np.mean(all_test_comp, axis=0)
    blend_raw = dual_track_w * final_main_raw + (1 - dual_track_w) * final_comp
    final_preds = np.clip(iso_calib.predict(blend_raw), 0, 1) if USE_ISOTONIC_CALIBRATION else blend_raw
else:
    final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.106115
1,594195,0.002410
2,594196,0.121262
3,594197,0.002410
4,594198,0.517895
